<a href="https://colab.research.google.com/github/shruti230844/sentiment--analysis/blob/main/phase1_230844.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1: Logistic Regression from Scratch
This notebook implements logistic regression for sentiment classification using a dataset different from the Coursera tweet dataset.

The goal is to:

* preprocess text
* extract features
* implement logistic regression from scratch
* evaluate performance
* perform error analysis

No external machine learning libraries such as sklearn are used.


In [1]:
import numpy as np
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
train.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [6]:
train['text'] = train['Title'] + " " + train['Description']
test['text'] = test['Title'] + " " + test['Description']
train_labels = train['Class Index'].values
test_labels = test['Class Index'].values
train_labels = np.where((train_labels==1) | (train_labels==3),1,0)
test_labels = np.where((test_labels==1) | (test_labels==3),1,0)
train_reviews = train['text'].values
test_reviews = test['text'].values

## Text Preprocessing

The `process_text()` function cleans the text by performing:

• Lowercasing
• Removing punctuation
• Tokenization
• Stopword removal
• Stemming

This helps reduce noise in the dataset and improves model performance.


In [7]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def process_text(review):

    review = review.lower()

    tokens = re.findall(r"[a-zA-Z]+|[!?]", review)

    words = []

    for word in tokens:
        if word not in stop_words:
            words.append(stemmer.stem(word))

    return words

In [8]:
process_text("I love this movie! It was amazing!")

['love', 'movi', '!', 'amaz', '!']

## Frequency Dictionary

The frequency dictionary stores how many times a word appears in positive and negative reviews.

Key format:
(word, label)

Example:
("good",1) → positive count
("bad",0) → negative count


In [9]:
def build_freqs(reviews, labels):

    freqs = {}

    for review, label in zip(reviews, labels):

        words = process_text(review)

        for word in words:

            pair = (word, label)

            if pair in freqs:
                freqs[pair] += 1
            else:
                freqs[pair] = 1

    return freqs

In [10]:
freqs = build_freqs(train_reviews, train_labels)

print(len(freqs))

59061


## Feature Extraction

Each review is converted into a 3-dimensional feature vector:

[1, positive_word_count, negative_word_count]

1 → bias term
positive_word_count → number of positive words
negative_word_count → number of negative words


In [11]:
def extract_features(review, freqs):

    words = process_text(review)

    x = np.zeros((1,3))

    x[0,0] = 1

    for word in words:

        x[0,1] += freqs.get((word,1),0)
        x[0,2] += freqs.get((word,0),0)

    return x


In [12]:
extract_features(train_reviews[0], freqs)

array([[1.0000e+00, 3.7749e+04, 1.8667e+04]])

## Sigmoid Function

The sigmoid function converts the linear regression output into a probability between 0 and 1.

Formula:

h(z) = 1 / (1 + e^-z)


In [13]:
def sigmoid(z):

    return 1/(1+np.exp(-z))

In [14]:
def compute_cost(X, y, theta):

    m = len(y)

    h = sigmoid(np.dot(X,theta))

    cost = -1/m * (np.dot(y.T,np.log(h)) + np.dot((1-y).T,np.log(1-h)))

    return cost

## Gradient Descent

Gradient descent updates the model parameters iteratively to minimize the cost function.

Learning rate used: α = 1e-9
Iterations: 1500


In [15]:
def gradient_descent(X, y, theta, alpha, num_iters):

    m = len(y)

    for i in range(num_iters):

        z = np.dot(X,theta)

        h = sigmoid(z)

        theta = theta - (alpha/m) * np.dot(X.T,(h-y))

    cost = compute_cost(X,y,theta)

    return theta, cost

In [16]:
X = np.zeros((len(train_reviews),3))

for i in range(len(train_reviews)):
    X[i,:] = extract_features(train_reviews[i], freqs)

y = train_labels.reshape(-1,1)

In [17]:
theta = np.zeros((3,1))

theta, cost = gradient_descent(X, y, theta, 1e-9, 1500)

print("Final cost:", cost)
print("Final theta:", theta)

Final cost: [[0.36565244]]
Final theta: [[ 5.47374700e-09]
 [ 1.51692140e-04]
 [-1.85335969e-04]]


In [18]:
def predict_review(review, freqs, theta):

    x = extract_features(review, freqs)

    y_pred = sigmoid(np.dot(x,theta))

    return y_pred

In [19]:
def test_logistic_regression(test_reviews, test_labels, freqs, theta):

    y_hat = []

    for review in test_reviews:

        pred = predict_review(review,freqs,theta)

        if pred > 0.5:
            y_hat.append(1)
        else:
            y_hat.append(0)

    accuracy = sum(np.array(y_hat)==test_labels)/len(test_labels)

    return accuracy

In [20]:
accuracy = test_logistic_regression(test_reviews,test_labels,freqs,theta)

print("Test accuracy:", accuracy)

Test accuracy: 0.8505263157894737


In [21]:
count = 0

for review,label in zip(test_reviews,test_labels):

    pred = predict_review(review,freqs,theta)

    if (pred>0.5) != label:

        print("Review:",review)
        print("True:",label)
        print("Predicted:",pred)
        print()

        count += 1

    if count == 5:
        break

Review: Loosing the War on Terrorism \\"Sven Jaschan, self-confessed author of the Netsky and Sasser viruses, is\responsible for 70 percent of virus infections in 2004, according to a six-month\virus roundup published Wednesday by antivirus company Sophos."\\"The 18-year-old Jaschan was taken into custody in Germany in May by police who\said he had admitted programming both the Netsky and Sasser worms, something\experts at Microsoft confirmed. (A Microsoft antivirus reward program led to the\teenager's arrest.) During the five months preceding Jaschan's capture, there\were at least 25 variants of Netsky and one of the port-scanning network worm\Sasser."\\"Graham Cluley, senior technology consultant at Sophos, said it was staggeri ...\\
True: 0
Predicted: [[0.61053047]]

Review: E-mail scam targets police chief Wiltshire Police warns about "phishing" after its fraud squad chief was targeted.
True: 0
Predicted: [[0.76437324]]

Review: Group to Propose New High-Speed Wireless Format  LOS 

## Error Analysis

### Misclassified Review 1

The article about terrorism was predicted as positive because the model relies only on word frequencies. Some neutral words in the sentence may appear frequently in positive training examples, causing the classifier to assign a higher positive score.

### Misclassified Review 2

The phishing scam article was incorrectly classified because the model does not understand context. Words like “targeted” or “warns” may appear in both positive and negative contexts, confusing the classifier.

### Misclassified Review 3

The technology-related article was misclassified because the bag-of-words features cannot capture the meaning of technical terms. Many technology words appear in multiple classes, making the model unable to distinguish sentiment.

### Misclassified Review 4

The dolphin research article contains mostly neutral scientific vocabulary. Since the model only counts word frequencies, it cannot determine the sentiment of factual statements.

### Misclassified Review 5

The dinosaur growth article is a long factual description with no strong sentiment words. Because the model depends on simple word counts, it struggles with long neutral sentences.
